In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable

In [0]:
BRONZE_PATH = "abfss://bronze@pravdatalake.dfs.core.windows.net"
SILVER_PATH = "abfss://silver@pravdatalake.dfs.core.windows.net"
SILVER_TABLE_PATH = f"{SILVER_PATH}/basic_table"
SILVER_TABLE_NAME = "vehicle_sales.silver.basic_table"

In [0]:
basic_df = spark.read.format("csv")\
    .option("header", True)\
    .option("inferSchema", True)\
    .load(f"{BRONZE_PATH}/Basic_table")
    

In [0]:
basic_df.display()

In [0]:
print(f"bronze row count: {basic_df.count()}")

In [0]:
silver_basic = (
    basic_df
    .withColumn("Genmodel_ID", trim(col("Genmodel_ID")))
    .withColumn("Automaker", trim(initcap(col("Automaker"))))
    .withColumn("Genmodel", trim(col("Genmodel")))
    .withColumn("Automaker_ID", col("Automaker_ID").cast(IntegerType()))
    .filter(col("Genmodel_ID").isNotNull())
    .filter(col("Automaker").isNotNull())
    .dropDuplicates(["Genmodel_ID"])
    .withColumn("silver_ingestion_timestamp", current_timestamp())
)

In [0]:
silver_basic.display()

####Data Quality Checks

In [0]:
row_count = silver_basic.count()


In [0]:
null_key_count = silver_basic.filter(col("Genmodel_ID").isNull()).count()

In [0]:
duplicate_key_count = silver_basic.groupBy("Genmodel_ID").count().filter("count > 1").count()

In [0]:
print(f"silver row count: {row_count}")
print(f"null Genmodel_ID count: {null_key_count}")
print(f"duplicate Genmodel_ID count: {duplicate_key_count}")

In [0]:
assert null_key_count == 0, "Genmodel_ID should never be null in silver_basic"
assert duplicate_key_count == 0, "Genmodel_ID should be unique in silver_basic"

In [0]:
if DeltaTable.isDeltaTable(spark, SILVER_TABLE_PATH):
 
    silver_table = DeltaTable.forPath(spark, SILVER_TABLE_PATH)
 
    (silver_table.alias("t")
        .merge(silver_basic.alias("s"), "t.Genmodel_ID = s.Genmodel_ID")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
 
else:
 
    silver_basic.write \
        .format("delta") \
        .mode("overwrite") \
        .save(SILVER_TABLE_PATH)

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {SILVER_TABLE_NAME}
    USING DELTA
    LOCATION '{SILVER_TABLE_PATH}'
""")

In [0]:
spark.sql(f"OPTIMIZE {SILVER_TABLE_NAME}")